# Week 2 — Exploratory Data Analysis & Visualization Walkthrough

UCI Online Retail Dataset (541,909 rows × 8 cols, Dec 2010 – Dec 2011)

This notebook walks through the complete EDA pipeline implemented in
`src/` and replicates the analyses that populate the final DOCX report.
Every number shown below is computed from the actual dataset.

## Step 0 — Setup & Imports

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
assert (PROJECT_ROOT / "src" / "main.py").exists(), "Run from the online-retail-eda/ root directory"

import numpy as np
import pandas as pd
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)

pd.set_option("display.max_columns", 25)
pd.set_option("display.float_format", "{:,.3f}".format)

print("Imports OK. Pandas", pd.__version__, "NumPy", np.__version__)

## Step 1 — Load & Validate Dataset

In [ ]:
from src.data_loading import load_excel_dataset
from src.data_validation import run_full_validation, count_missing_values

df_raw = load_excel_dataset()
validation = run_full_validation(df_raw)
missing_df = count_missing_values(df_raw)

print(f"Rows = {len(df_raw):,}  Cols = {df_raw.shape[1]}  Memory ≈ {df_raw.memory_usage(deep=True).sum()/1024/1024:.1f} MB")
print(f"Invoices = {validation['n_invoices']:,}   Customers = {validation['n_customers']:,}")
print(f"Products = {validation['n_products']:,}   Countries = {validation['n_countries']}")
missing_df

## Step 2 — Feature Engineering & Analytical Datasets

We maintain **three** analytical datasets so every exclusion is explicit:

| Dataset | Filters | Use |
|---|---|---|
| `full` | none | Missing values, duplicates, raw frequencies |
| `valid_price` | UnitPrice > 0 | Price-aware analysis |
| `positive_sales` | valid_price + ¬cancellation + Quantity > 0 | Revenue / sales analyses |

In [ ]:
from src.data_preparation import prepare_all, numeric_summary

datasets = prepare_all(df_raw)
df = datasets["engineered"]       # full dataset with features added
df_pos = datasets["positive_sales"]

print(f"full rows:          {len(datasets['full']):>10,}")
print(f"valid_price rows:   {len(datasets['valid_price']):>10,}")
print(f"positive_sales rows:{len(df_pos):>10,}")
print()
print("Columns now available:", sorted(df.columns.tolist()))
print()
numeric_summary(df_pos, columns=["Quantity", "UnitPrice", "TotalAmount"])

## Step 3 — Descriptive Analysis (overview, frequencies)

In [ ]:
from src.descriptive_analysis import run_descriptive_analysis
desc = run_descriptive_analysis(df, df_pos)
print("Top 5 countries by lines:")
desc["cat_freqs"]["Country"].head(5)

## Step 4 — Visualization: Missing Values Bar Chart

In [ ]:
from src.visualization import plot_missing_values
plot_missing_values(missing_df, filename="nb_01_missing.png")

In [ ]:
from IPython.display import Image, display
display(Image(filename="outputs/figures/nb_01_missing.png", width=800))

## Step 5 — Temporal Analysis (monthly volume, revenue, hourly, weekday)

In [ ]:
from src.temporal_analysis import run_temporal_analysis
from src.visualization import (
    plot_monthly_transactions, plot_monthly_revenue,
    plot_transactions_by_hour, plot_transactions_by_weekday,
)

temp = run_temporal_analysis(df, df_pos)
print("Monthly volume head (sorted):")
temp["monthly_volume"].head(13)

In [ ]:
plot_monthly_transactions(temp["monthly_volume"], filename="nb_04_monthly_vol.png")
display(Image(filename="outputs/figures/nb_04_monthly_vol.png", width=900))

## Step 6 — Product Analysis (Quantity vs Revenue rankings)

In [ ]:
from src.product_analysis import run_product_analysis
from src.visualization import plot_top_products_quantity, plot_top_products_revenue

prod = run_product_analysis(df_pos)

print("Top 5 by Quantity (rows shown = 20 saved):")
display(prod["top_quantity"].head(5))
print()
print("Top 5 by Revenue:")
display(prod["top_revenue"].head(5))
print()
print("Ranking comparison (Top 10):")
display(prod["ranking_comparison"])

In [ ]:
plot_top_products_quantity(prod["top_quantity"])
display(Image(filename="outputs/figures/07_top_products_quantity.png", width=900))

## Step 7 — Geographic Analysis

In [ ]:
from src.geographic_analysis import run_geographic_analysis
from src.visualization import plot_top_countries

geo = run_geographic_analysis(df, df_pos)
print("Top 10 countries by lines:")
display(geo["country_transactions"].head(10))

plot_top_countries(geo["country_transactions"])
display(Image(filename="outputs/figures/06_top_countries.png", width=900))

## Step 8 — Correlation Analysis (3×3 matrix + two scatter studies)

In [ ]:
from src.relationship_analysis import run_relationship_analysis
from src.visualization import (
    plot_correlation_heatmap,
    plot_quantity_vs_totalamount,
    plot_unitprice_vs_quantity,
)

rel = run_relationship_analysis(df_pos)
print("Pearson Correlation Matrix:")
display(rel["correlation_matrix"])
print()
print(f"Qty vs Amt  r = {rel['qty_vs_totalamount']['correlation']:.4f}  Spearman = {rel['qty_vs_totalamount']['spearman']:.4f}")
print(f"Price vs Qty r = {rel['unitprice_vs_quantity']['correlation']:.4f}  Spearman = {rel['unitprice_vs_quantity']['spearman']:.4f}")

plot_correlation_heatmap(rel["correlation_matrix"], filename="nb_13_heatmap.png")
display(Image(filename="outputs/figures/nb_13_heatmap.png", width=650))

## Step 9 — Anomaly Analysis (IQR + Concrete Real-Data Examples + Tail Percentiles)

In [ ]:
from src.anomaly_analysis import run_anomaly_analysis

anom = run_anomaly_analysis(df, df_pos)
print("IQR Outlier Summary:")
display(anom["iqr_summary"])
print()
print("Heavy-tail Percentile Analysis:")
display(anom["percentile_analysis"])
print()
print("Top-3 Largest-Qty positive-sales transactions (actual rows!):")
examples = anom["examples"]
cols = [c for c in ["InvoiceNo","StockCode","Description","Quantity","UnitPrice","TotalAmount","Country"] if c in examples["largest_quantity"].columns]
display(examples["largest_quantity"][cols].head(3))

## Step 10 — Distributions: Quantity, UnitPrice, TotalAmount (log1p hist + box)

In [ ]:
from src.visualization import (
    plot_quantity_distribution,
    plot_unitprice_distribution,
    plot_totalamount_distribution,
)

qh, qb = plot_quantity_distribution(df_pos)
ph, pb = plot_unitprice_distribution(df_pos)
ah, ab = plot_totalamount_distribution(df_pos)

print(f"Quantity hist -> {qh}")
print(f"UnitPrice hist -> {ph}")
print(f"TotalAmount hist -> {ah}")
display(Image(filename="outputs/figures/02_quantity_distribution.png", width=900))
display(Image(filename="outputs/figures/03_unit_price_distribution.png", width=900))

## Step 11 — Reproducible Run: Full Pipeline via `main.py`

The cell below runs the exact same pipeline as `python -m src.main` and 

In [ ]:
# Full pipeline run — uncomment to reproduce:
# from src.main import main as run_pipeline
# result = run_pipeline()
# print("Report:", result["report_path"])
# print("Figures generated:", len(result["figures"]))

## Step 12 — Key Findings (all computed from actual data)

To confirm all numbers below are real, run each analysis cell above and compare.

In [ ]:
iqr = anom["iqr_summary"].set_index("column")
findings = [
    f"1. Dataset rows = {len(df_raw):,} | Invoices = {validation['n_invoices']:,} | Customers = {validation['n_customers']:,} | Products = {validation['n_products']:,}",
    f"2. Missing: CustomerID has {int(missing_df.loc[missing_df.Column=='CustomerID','Missing Count'].iloc[0]):,} nulls ({float(missing_df.loc[missing_df.Column=='CustomerID','Missing Percentage (%)'].iloc[0]):.2f}%)",
    f"3. UK share of transaction lines = {float(geo['country_transactions'].iloc[0]['lines_pct']):.2f}%",
    f"4. IQR outliers: UnitPrice {int(iqr.loc['UnitPrice','outlier_count']):,} ({float(iqr.loc['UnitPrice','outlier_pct']):.2f}%) | Quantity {int(iqr.loc['Quantity','outlier_count']):,} | TotalAmount {int(iqr.loc['TotalAmount','outlier_count']):,}",
    f"5. Correlation: Qty vs Amt r = {rel['qty_vs_totalamount']['correlation']:.3f} | UnitPrice vs Qty r = {rel['unitprice_vs_quantity']['correlation']:.3f}",
]
for f in findings:
    print(f)

---

### End of notebook.

Final deliverables after running `python -m src.main`:
- 13+ high-res PNG figures → `outputs/figures/`
- 21+ CSV tables → `outputs/tables/`
- 12-section Word report → `report/Week_2_EDA_and_Visualization_Report.docx`
- All 17 tests pass → `pytest tests/`